# IACC Analytics Agent - Demo EP2
**Asignatura:** ISY0101 - Ingenieria de Soluciones con IA  
**Estudiante:** Robinson Arriagada Borquez  
**Modulos evaluados:** IL2.1 - IL2.2 - IL2.3 - IL2.4

Este notebook presenta tres escenarios de prueba que evidencian el comportamiento del agente segun los indicadores de evaluacion del modulo RA2.

| Escenario | Tipo de consulta | Herramientas activadas |
|-----------|-----------------|------------------------|
| 1 | Consulta puntual simple | clasificar_consulta -> faiss_retriever |
| 2 | Calculo de KPI con datos provistos | clasificar_consulta -> calcular_kpi |
| 3 | Conversacion multi-turno con referencia implicita | clasificar_consulta -> faiss_retriever + memoria |


## 0. Instalacion y configuracion

In [ ]:
# Instalar dependencias (ejecutar solo una vez)
# !pip install langchain langchain-openai langchain-community faiss-cpu openai -q

In [ ]:
import os

# Configurar variables de entorno
# Reemplazar con el token real antes de ejecutar
# os.environ['GITHUB_TOKEN']    = 'ghp_TU_TOKEN_AQUI'
# os.environ['GITHUB_BASE_URL'] = 'https://models.inference.ai.azure.com'

os.environ.setdefault('OPENAI_API_KEY',  os.getenv('GITHUB_TOKEN', ''))
os.environ.setdefault('OPENAI_API_BASE', os.getenv(
    'GITHUB_BASE_URL', 'https://models.inference.ai.azure.com'))

print('Variables de entorno configuradas.')

In [ ]:
# Importar el agente desde agent_ep2.py
# El archivo agent_ep2.py debe estar en el mismo directorio que este notebook
from agent_ep2 import IACCAnalyticsAgent, _get_vectorstore

# Pre-cargar el indice FAISS antes de la primera consulta
_get_vectorstore()
print('Agente e indice FAISS listos.')

---
## Escenario 1 - Consulta puntual simple

**Objetivo:** Verificar que el agente recupera datos correctos del vectorstore y los presenta citando el periodo.

**Flujo ReAct esperado:**
1. `clasificar_consulta` identifica tipo CONSULTA_PUNTUAL
2. `faiss_retriever` recupera chunks de PEM-2025-01 con k=3
3. El LLM genera la respuesta usando unicamente el contexto recuperado

In [ ]:
agente = IACCAnalyticsAgent(verbose=True)

print('=' * 60)
print('ESCENARIO 1 - Consulta puntual')
print('=' * 60)

respuesta_1 = agente.consultar(
    'Cual fue la tasa de conversion total en el periodo PEM-2025-01?'
)
print('\nRESPUESTA FINAL:\n')
print(respuesta_1)

**Evidencia IL2.1:** El agente selecciono las herramientas correctas en secuencia sin instruccion explicita del usuario.  
**Evidencia IL2.3:** Clasifico la consulta como PUNTUAL y ajusto k=3 en la busqueda semantica.

---
## Escenario 2 - Calculo de KPI con condicion cambiante

**Objetivo:** Demostrar toma de decisiones adaptativa. El agente detecta intencion de calculo y activa `calcular_kpi` en lugar de `faiss_retriever`, ajustando su comportamiento segun la naturaleza de la consulta.

**Flujo ReAct esperado:**
1. `clasificar_consulta` identifica tipo CALCULO_KPI
2. `calcular_kpi` recibe los valores extraidos del lenguaje natural
3. El LLM interpreta el resultado e incluye la clasificacion de eficiencia

In [ ]:
print('=' * 60)
print('ESCENARIO 2 - Calculo de KPI')
print('=' * 60)

respuesta_2 = agente.consultar(
    'Calcula la tasa de conversion si tuvieramos 850 leads y 130 matriculas. '
    'El tiempo de cierre promedio fue de 12 dias.'
)
print('\nRESPUESTA FINAL:\n')
print(respuesta_2)

**Evidencia IL2.1:** El agente uso `calcular_kpi` de forma autonoma extrayendo los parametros numericos del lenguaje natural.  
**Evidencia IL2.3:** Selecciono una herramienta distinta al Escenario 1 segun la intencion detectada, lo que demuestra comportamiento adaptativo.

---
## Escenario 3 - Conversacion multi-turno con memoria

**Objetivo:** Demostrar que el agente mantiene coherencia entre turnos consecutivos usando ConversationBufferWindowMemory.

**Flujo esperado:**
- Turno 1: consulta sobre el Grupo Pro
- Turno 2: referencia implicita ('cuantos dias tardaron') que el agente debe resolver usando el historial
- Turno 3: comparacion con el equipo contrario, lo que activa una busqueda ampliada

In [ ]:
print('=' * 60)
print('ESCENARIO 3 - Multi-turno con memoria')
print('=' * 60)

# Turno 1
print('\n-- TURNO 1 --')
respuesta_3a = agente.consultar(
    'Como se desempeno el Grupo Pro en PEM-2025-01?'
)
print('\nRESPUESTA TURNO 1:\n')
print(respuesta_3a)

# Turno 2: referencia implicita al contexto anterior
print('\n-- TURNO 2 (referencia implicita) --')
respuesta_3b = agente.consultar(
    'Y cuantos dias tardaron en cerrar sus matriculas?'
)
print('\nRESPUESTA TURNO 2:\n')
print(respuesta_3b)

# Turno 3: comparacion con el otro equipo
print('\n-- TURNO 3 (comparacion) --')
respuesta_3c = agente.consultar(
    'Eso es mejor que el Grupo Aleatorio?'
)
print('\nRESPUESTA TURNO 3:\n')
print(respuesta_3c)

**Evidencia IL2.2:** En el Turno 2, el agente resolvio la referencia implicita 'sus matriculas' usando el historial sin que el usuario repitiera el nombre del equipo.  
**Evidencia IL2.3:** En el Turno 3, clasifico la consulta como COMPARACION y amplio la busqueda para recuperar datos de ambos equipos.

---
## Inspeccion del historial de memoria

In [ ]:
# Revisar el estado de la memoria de corto plazo
historial = agente.memory.load_memory_variables({})
print('Historial en memoria (ConversationBufferWindowMemory, ultimos 5 turnos):')
for msg in historial.get('chat_history', []):
    rol = 'Usuario' if msg.type == 'human' else 'Agente'
    print(f'  [{rol}]: {str(msg.content)[:120]}...')

---
## Resumen de evidencias por Indicador de Evaluacion

| IE | Componente demostrado | Escenario |
|----|----------------------|-----------|
| IE1 | Las herramientas ejecutan funciones especificas con autonomia | 1, 2, 3 |
| IE2 | LangChain AgentExecutor como framework escalable con ReAct | Todos |
| IE3 | ConversationBufferWindowMemory mantiene coherencia multi-turno | 3 |
| IE4 | FAISS recupera contexto semantico relevante en cada consulta | 1, 3 |
| IE5 | clasificar_consulta secuencia y prioriza el flujo de tareas | Todos |
| IE6 | Los tres escenarios muestran comportamiento adaptativo ante entradas distintas | 1, 2, 3 |
